# SBI2 hackathon: download the participant datasets

Use this notebook to choose, preview, and download the processed datasets for the hackathon. It follows the Altair starter notebook workflow: first inspect what will be downloaded, then run the download, then inspect the resulting profiles.

Before opening the notebook, install the repository environment with `python -m pip install -r requirements.txt` from the repository root.

## Available datasets

| Selection | Contents | Approximate size |
|---|---|---:|
| `altair-phase` | A549 guide-level phase profiles and small metadata | 45.8 MiB plus metadata |
| `altair-4i` | 10 A549 Biohub_OPS0077 4i guide-by-screen profiles | 91.9 MiB extracted; requires downloading the 1.90 GiB Altair marker archive |
| `posh` | POSH published 124-gene morphology profile table | 4.47 MiB |
| `aconcagua` | HeLa interphase and mitotic aggregate profiles plus metadata | 12.1 MiB |

The four-group download is about 1.96 GiB of network transfer, plus small metadata files. The 4i guide profiles are extracted from `markers_guide.zip`; the ZIP has no standalone member URLs. The optional sampled cell-level 4i files are much larger and are not included. Aconcagua `all` is not published as a processed profile, so both available populations are downloaded. No raw images or Aconcagua cell-level Parquet are included.

POSH data are published under CC BY-NC-SA 4.0. Confirm the source terms for the event and any redistribution. Keep downloaded data out of GitHub.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / 'scripts' / 'download_data.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from the sbi2-hackathon-prep-data-ops repository.')

# On the VM, set SBI2_DATA_DIR=/mnt/hackathon-data before starting Jupyter.
DATA_DIR = Path(os.environ.get('SBI2_DATA_DIR', ROOT / 'data')).expanduser().resolve()
DATASET_CHOICES = ('altair-phase', 'altair-4i', 'posh', 'aconcagua')
DATASETS_TO_DOWNLOAD = ['altair-phase', 'altair-4i', 'posh', 'aconcagua']

# Set to True in the download cell below only after reviewing the size preview.
DOWNLOAD_NOW = False
print('Repository:', ROOT)
print('Data directory:', DATA_DIR)
print('Selected:', DATASETS_TO_DOWNLOAD)

## 1. Preview the selected download

Edit `DATASETS_TO_DOWNLOAD` above to select only what you need. This cell queries current source sizes and writes no data files. Keep `altair-4i` selected for the Biohub_OPS0077 4i profiles; the preview will show the full marker-archive transfer size.

In [ ]:
unknown = set(DATASETS_TO_DOWNLOAD) - set(DATASET_CHOICES)
if unknown:
    raise ValueError(f'Unknown dataset selections: {sorted(unknown)}')
if not DATASETS_TO_DOWNLOAD:
    raise ValueError('Select at least one dataset group.')

download_script = ROOT / 'scripts' / 'download_data.py'
preview_cmd = [sys.executable, str(download_script), '--datasets', *DATASETS_TO_DOWNLOAD, '--data-dir', str(DATA_DIR), '--dry-run']
preview = subprocess.run(preview_cmd, cwd=ROOT, text=True, capture_output=True)
print(preview.stdout)
if preview.stderr:
    print(preview.stderr)
if preview.returncode:
    raise RuntimeError(f'Download preview failed with exit code {preview.returncode}')

## 2. Download the selected data

After reviewing the preview, set `DOWNLOAD_NOW = True` in the setup cell and run this cell. The downloader verifies known SHA-256 hashes, checks object sizes, writes files atomically, extracts the 10 OPS0077 profiles, and saves `download_manifest.json`. Re-running it reuses verified existing files.

In [ ]:
if DOWNLOAD_NOW:
    download_cmd = [sys.executable, str(download_script), '--datasets', *DATASETS_TO_DOWNLOAD, '--data-dir', str(DATA_DIR), '--yes']
    subprocess.run(download_cmd, cwd=ROOT, check=True)
else:
    print('Download skipped. Review the preview, set DOWNLOAD_NOW = True, then rerun this cell.')

## 3. Review files and run first-pass quality checks

The quality report checks file hashes, H5AD and Parquet readability, profile dimensions, duplicate identifiers, metadata columns, and guide overlap between OPS0077 and phase when both are selected. Missing groups are listed when you intentionally download only a subset. These checks do not establish biological validity or independently verify the originating guide library.

In [ ]:
manifest_path = DATA_DIR / 'download_manifest.json'
if not manifest_path.is_file():
    print('No download manifest found. Run the download cell first.')
else:
    import json
    manifest = json.loads(manifest_path.read_text())
    for item in manifest['files']:
        print(f"{item['size_bytes'] / 1024**2:8.2f} MiB  {item['path']}  sha256={item['sha256'][:12]}…")

    qc_cmd = [sys.executable, str(ROOT / 'scripts' / 'run_quality_checks.py'), '--data-dir', str(DATA_DIR), '--outdir', str(DATA_DIR / 'qc'), '--allow-incomplete']
    qc = subprocess.run(qc_cmd, cwd=ROOT, text=True, capture_output=True)
    print(qc.stdout)
    if qc.stderr:
        print(qc.stderr)
    if qc.returncode:
        report_path = DATA_DIR / 'qc' / 'quality_report.json'
        raise RuntimeError(f'Quality checks reported an error; inspect {report_path}')

## Where the files go

```text
data/
  altair/phase_guide.h5ad
  altair/ops0077_4i/*_guide_x_screen.h5ad
  posh/morphology_124.pq
  aconcagua/interphase_aggregated_data.h5ad
  aconcagua/mitotic_aggregated_data.h5ad
  download_manifest.json
  qc/quality_report.json
```

See `docs/DATASETS.md` for detailed contents and `docs/AWS_VM.md` for staging to shared EBS storage.